<a href="https://colab.research.google.com/github/muhammadrakayansyah-beep/afl-raka-/blob/main/AIF_P2_25120200002_Muhammad_Rakah_Yansyah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI in Finance / Pertemuan 2
## Membaca kolom dan memahami rasio pengajuan pinjaman
Dosen: **Haikal Shiddiq S.Kom., M.T.**  
Universitas Cakrawala / Program Studi Ekonomi, Keuangan, dan Perbankan / Blok B 2026/2027  
Revisi materi 12 November 2026

### Mulai di sini
1. Buka notebook dari tautan pada LMS kelas di Google Colab.
2. Pilih **File > Save a copy in Drive**. Kerjakan salinan pribadi, bukan master dosen.
3. Ubah nama salinan menjadi **AIF_P2_NIM_Nama**. Isi identitas di bawah.
4. Klik **Connect**, lalu jalankan setiap cell dari atas ke bawah dengan tombol segitiga.
5. Pada bagian formulir, ubah angka atau pilihan lalu klik segitiga lagi. Output mengikuti eksekusi terakhir.
6. Isi bagian **Jawaban Anda** dengan membuka cell teks, kemudian simpan. Bagikan salinan kepada email dosen pada LMS sebagai **Viewer**, dan kirim tautan salinan tersebut ke LMS.

Gunakan runtime CPU standar. Latihan kecil ini tidak memerlukan GPU, API key, instalasi Jupyter lokal, atau berkas data tambahan. Data fiktif tersedia secara sengaja di notebook. Semua hasil untuk pembelajaran, bukan keputusan investasi/kredit nyata.

### Python dalam bahasa finance
Python adalah bahasa pemrograman: kita menuliskan langkah agar komputer membaca angka, menghitung rumus, dan menampilkan hasil. Contoh `cash_in - cash_out` mirip rumus selisih di spreadsheet. Fokus Anda adalah makna angka dan keputusan yang didukung, bukan menghafal syntax.
Colab menyediakan tempat menjalankan Python di cloud. Browser menampilkan code dan output; komputer cloud mengerjakan perhitungannya.

Jadwal WIB: AiFn2 Kamis P1 18.00–20.00 dan P2 20.00–22.00 (jeda 20.00–20.10). AiFn1 Jumat P1 08.30–10.00 dan P2 10.00–11.30 (peregangan 10.00–10.03).

Jika perangkat atau koneksi terkendala, kerjakan lembar cadangan individual terlebih dahulu. Catat kendalanya kepada dosen. Eksekusi salinan pribadi tetap perlu dilengkapi pada waktu yang ditetapkan dosen.


In [1]:
# @title Identitas mahasiswa
nama = "Muhammad Rakah Yansyah" # @param {type:"string"}
nim = "25120200002" # @param {type:"string"}
kelas = "AiFn2" # @param ["AiFn2", "AiFn1"]
print(f"Nama: {nama} | NIM: {nim} | Kelas: {kelas}")


Nama: Muhammad Rakah Yansyah | NIM: 25120200002 | Kelas: AiFn2


In [2]:
# @title Persiapan otomatis
import sys
import pandas as pd
import matplotlib.pyplot as plt
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("Siap. Semua data latihan bersifat fiktif.")


Python: 3.13.18
pandas: 2.2.3
Siap. Semua data latihan bersifat fiktif.


## Praktikum 1 / Tabel yang akan dibaca
Dataset fiktif berisi **8 baris dan 3 kolom**. Data sengaja tersedia di cell berikut supaya Anda tidak perlu upload CSV atau menghubungkan seluruh Google Drive ke runtime.

- **id_pengajuan:** identitas catatan, misalnya A001; berupa teks.
- **pendapatan_bulanan:** pendapatan dalam rupiah per bulan.
- **cicilan_bulanan:** cicilan dalam rupiah per bulan.

Satu **row/baris** berarti satu catatan. Satu **column/kolom** berarti satu jenis informasi. Jumlah baris belum menjamin jumlah pengajuan unik, karena satu pengajuan dapat tercatat dua kali.


In [3]:
# @title Baca data yang sudah tersedia
from io import StringIO
DATA_LATIHAN = 'id_pengajuan,pendapatan_bulanan,cicilan_bulanan\nA001,8000000,2000000\nA002,6000000,1800000\nA003,,1500000\nA004,10000000,4500000\nA002,6000000,1800000\nA005,0,800000\nA006,7000000,-200000\nA007,5000000,2000000\n'
df = pd.read_csv(StringIO(DATA_LATIHAN), dtype={"id_pengajuan": "string"})
print("Jumlah baris dan kolom:", df.shape)
print("Nama kolom:", list(df.columns))
print(df.to_string(index=False))
print("Nilai kosong per kolom:")
print(df.isna().sum())
print("Salinan identik:", int(df.duplicated().sum()))


Jumlah baris dan kolom: (8, 3)
Nama kolom: ['id_pengajuan', 'pendapatan_bulanan', 'cicilan_bulanan']
id_pengajuan  pendapatan_bulanan  cicilan_bulanan
        A001           8000000.0          2000000
        A002           6000000.0          1800000
        A003                 NaN          1500000
        A004          10000000.0          4500000
        A002           6000000.0          1800000
        A005                 0.0           800000
        A006           7000000.0          -200000
        A007           5000000.0          2000000
Nilai kosong per kolom:
id_pengajuan          0
pendapatan_bulanan    1
cicilan_bulanan       0
dtype: int64
Salinan identik: 1


### Cara membaca hasil
`(8, 3)` berarti 8 baris dan 3 kolom. A002 memiliki satu salinan identik. `NaN` pada pendapatan A003 menunjukkan nilai yang belum tersedia; maknanya berbeda dari angka nol pada A005.

Pada sesi ini kita mencatat masalah dan memahami konteks. Keputusan menghapus, mengisi, atau memperbaiki data dilanjutkan pada sesi 3.

## Praktikum 2 / Rasio cicilan
Rumus: **cicilan bulanan / pendapatan bulanan**. Angka `0.25` berarti `25%`.

1. Pilih **A001**, biarkan **gunakan_skenario** tidak dicentang, lalu jalankan: rasio awal 25%.
2. Centang **gunakan_skenario**, isi **cicilan_skenario_juta = 2.8**, lalu jalankan: rasio 35% pada pendapatan tetap Rp8 juta.
3. Pilih **A003**, **A005**, lalu **A006** secara bergantian. Amati mengapa perhitungan memerlukan klarifikasi.

Skenario tidak menimpa data sumber. Nilai kosong, nol pada pembagi, dan cicilan negatif tidak langsung dipakai untuk mengambil keputusan kredit.


In [4]:
# @title Pilih catatan dan ubah skenario cicilan
id_diperiksa = "A001" # @param ["A001", "A002", "A003", "A004", "A005", "A006", "A007"]
gunakan_skenario = True # @param {type:"boolean"}
cicilan_skenario_juta = 2.8 # @param {type:"number"}

pilihan = df.loc[df["id_pengajuan"] == id_diperiksa]
catatan = pilihan.iloc[0]
pendapatan = catatan["pendapatan_bulanan"]
cicilan = catatan["cicilan_bulanan"]
rasio_awal = None
rasio_skenario = None
if pd.isna(pendapatan) or pd.isna(cicilan):
    pesan = "Perlu klarifikasi: pendapatan atau cicilan belum tersedia."
elif pendapatan <= 0:
    pesan = "Perlu klarifikasi: pendapatan harus positif untuk menjadi pembagi rasio."
elif cicilan < 0:
    pesan = "Perlu klarifikasi: cicilan negatif tidak sesuai definisi latihan."
elif gunakan_skenario and cicilan_skenario_juta < 0:
    pesan = "Perlu klarifikasi: skenario cicilan tidak boleh negatif."
else:
    rasio_awal = cicilan / pendapatan
    rasio_skenario = cicilan_skenario_juta * 1_000_000 / pendapatan if gunakan_skenario else rasio_awal
    pesan = f"Rasio awal {rasio_awal:.0%}. Rasio pada skenario terpilih {rasio_skenario:.0%}."
print("Catatan:", id_diperiksa)
if len(pilihan) > 1:
    print("Ada salinan identik. Contoh ini menelaah catatan pertama; data sumber belum dihapus.")
print(pesan)
print("Ini perhitungan formula, belum proses training ML dan belum keputusan kredit.")


Catatan: A001
Rasio awal 25%. Rasio pada skenario terpilih 35%.
Ini perhitungan formula, belum proses training ML dan belum keputusan kredit.


### Arti code tanpa menghafal syntax
- `pd.read_csv(...)` membaca data berbentuk CSV menjadi tabel pandas.
- `df.shape` menunjukkan jumlah baris dan kolom.
- `df.loc[...]` memilih catatan yang ingin diperiksa.
- `if` memeriksa kondisi sebelum menghitung.
- `cicilan / pendapatan` menghitung rasio. Tampilan persen membantu membacanya.

### Jika output tampak lama
Mengubah formulir belum menjalankan perhitungan. Klik tombol segitiga pada cell yang diubah. Jika runtime terputus atau muncul `NameError`, hubungkan kembali dan jalankan cell dari atas, atau pilih **Runtime > Run all**. Simpan kembali salinan di Drive. Code dan output yang tersimpan dapat dibaca dosen; memori runtime dapat berakhir.

In [5]:
# @title Rekap bukti: hasil awal, skenario, dan catatan yang perlu klarifikasi
# Cell tambahan mahasiswa. Logika pemeriksaan sama dengan cell formulir; df tidak diubah atau dihapus.
print("Bentuk tabel (baris, kolom):", df.shape)
print("Jumlah ID berbeda:", df["id_pengajuan"].nunique(), "| ID yang muncul lebih dari sekali:",
      df.loc[df["id_pengajuan"].duplicated(), "id_pengajuan"].tolist())
print("Tipe data:", dict(df.dtypes.astype(str)))

def periksa(id_x, skenario=False, cicilan_juta=2.8):
    r = df.loc[df["id_pengajuan"] == id_x].iloc[0]
    p, k = r["pendapatan_bulanan"], r["cicilan_bulanan"]
    if pd.isna(p) or pd.isna(k):
        return p, k, "Perlu klarifikasi: pendapatan/cicilan belum tersedia (NaN)"
    if p <= 0:
        return p, k, "Perlu klarifikasi: pendapatan nol/negatif, tidak bisa jadi pembagi"
    if k < 0:
        return p, k, "Perlu klarifikasi: cicilan negatif tidak sesuai definisi"
    if skenario:
        return p, cicilan_juta * 1_000_000, f"Rasio skenario {cicilan_juta*1_000_000/p:.0%}"
    return p, k, f"Rasio awal {k/p:.0%}"

baris = [("A001", "awal", *periksa("A001")),
         ("A001", "skenario 2,8 juta", *periksa("A001", True)),
         ("A003", "awal", *periksa("A003")),
         ("A005", "awal", *periksa("A005")),
         ("A006", "awal", *periksa("A006"))]
rekap = pd.DataFrame(baris, columns=["ID", "kondisi", "pendapatan (Rp/bln)", "cicilan (Rp/bln)", "hasil"])
print(rekap.to_string(index=False))
print("Baris di df setelah rekap:", len(df), "(data sumber tetap utuh)")


Bentuk tabel (baris, kolom): (8, 3)
Jumlah ID berbeda: 7 | ID yang muncul lebih dari sekali: ['A002']
Tipe data: {'id_pengajuan': 'string', 'pendapatan_bulanan': 'float64', 'cicilan_bulanan': 'int64'}
  ID           kondisi  pendapatan (Rp/bln)  cicilan (Rp/bln)                                                              hasil
A001              awal            8000000.0         2000000.0                                                     Rasio awal 25%
A001 skenario 2,8 juta            8000000.0         2800000.0                                                 Rasio skenario 35%
A003              awal                  NaN         1500000.0         Perlu klarifikasi: pendapatan/cicilan belum tersedia (NaN)
A005              awal                  0.0          800000.0 Perlu klarifikasi: pendapatan nol/negatif, tidak bisa jadi pembagi
A006              awal            7000000.0         -200000.0           Perlu klarifikasi: cicilan negatif tidak sesuai definisi
Baris di df setelah rekap

## Tugas praktik P2 / Jawaban Anda

- **Nama / NIM / kelas:** Muhammad Rakah Yansyah / 25120200002 / AiFn2

**A. Membaca tabel dan skenario**

- **Apa arti 8 baris dan 3 kolom? Satuan kolom nominal:** `df.shape = (8, 3)`. Ada 8 baris, artinya 8 *catatan* data, dan 3 kolom, artinya 3 jenis informasi untuk tiap catatan. Ketiga kolomnya: `id_pengajuan` (identitas catatan, berupa **teks**, misalnya A001, bukan angka untuk dihitung), `pendapatan_bulanan` dan `cicilan_bulanan`, keduanya dalam **rupiah per bulan**. Jumlah baris (8) tidak sama dengan jumlah pengajuan unik (**7**), karena **A002 muncul dua kali sebagai salinan identik** (`Salinan identik: 1`). Baris ganda itu dicatat saja, tidak dihapus.
- **Hasil A001 awal dan skenario cicilan Rp2,8 juta:** Rumus rasio = cicilan bulanan ÷ pendapatan bulanan.
  - Awal: 2.000.000 ÷ 8.000.000 = 0,25 = **25%**
  - Skenario: formulir memakai satuan juta, jadi 2,8 × 1.000.000 = 2.800.000. Hasilnya 2.800.000 ÷ 8.000.000 = 0,35 = **35%**
  - Perubahan: cicilan naik Rp800 ribu (+40%) sementara pendapatan tetap, sehingga rasio naik 10 poin persentase, dari 25% ke 35%. Artinya porsi pendapatan bulanan yang terpakai untuk cicilan bertambah dari seperempat menjadi lebih dari sepertiga. Skenario ini hanya simulasi dan tidak menimpa data sumber; cicilan A001 di tabel tetap 2.000.000.

**B. Catatan yang perlu klarifikasi**

| ID | Data | Masalah | Pertanyaan klarifikasi |
|---|---|---|---|
| A003 | pendapatan kosong (NaN), cicilan 1.500.000 | Nilai **belum tersedia**. Kita tidak tahu berapa pendapatannya. | "Berapa pendapatan bulanan A003, dan apa dokumen pendukungnya (slip gaji/rekening)? Mengapa kolom ini kosong?" |
| A005 | pendapatan **0**, cicilan 800.000 | Nilainya **ada dan tercatat nol**. Nol tidak bisa jadi pembagi (pembagian dengan nol tidak terdefinisi). | "Apakah pendapatan A005 memang Rp0 (misalnya belum bekerja), atau salah input? Jika memang nol, sumber dana apa yang dipakai untuk membayar cicilan Rp800 ribu?" |
| A006 | pendapatan 7.000.000, cicilan **−200.000** | **Cicilan negatif** tidak sesuai definisi cicilan (kewajiban bayar ≥ 0). | "Apakah −200.000 salah ketik untuk 200.000, atau ini koreksi/refund/kelebihan bayar yang tercatat di kolom cicilan? Berapa cicilan bulanan yang benar?" |

- **Mengapa A003 dan A005 perlu klarifikasi dengan alasan berbeda?** A003 = **data hilang** (NaN): informasinya belum ada, jadi tidak boleh dianggap nol. A005 = **data ada, bernilai nol**: secara matematis tidak bisa jadi pembagi, dan secara bisnis janggal karena ada cicilan tanpa pendapatan. Menyamakan keduanya, misalnya mengisi NaN dengan 0, akan mengubah arti data.
- **Mengapa A006 belum dipakai menghitung rasio?** Kalau dihitung apa adanya, hasilnya −200.000 ÷ 7.000.000 ≈ −2,9%, rasio negatif yang tidak punya arti ekonomi. Penyebab angka negatif belum diketahui (salah tanda atau transaksi lain), jadi rasionya ditahan sampai ada klarifikasi. Nilainya tidak dikarang (misalnya diubah sendiri menjadi 200.000) dan barisnya tidak dihapus.
- **Mengapa menghitung rasio belum sama dengan training ML?** Rasio dihitung dengan **rumus tetap yang ditentukan manusia** (cicilan ÷ pendapatan). Tidak ada contoh berlabel (misalnya riwayat terlambat/tidak terlambat), dan tidak ada proses mencari pola, memilih parameter, atau mengukur kesalahan terhadap data. Training ML berarti program mempelajari aturan/parameter dari data historis yang hasilnya sudah diketahui, seperti pemilihan batas 35%/45% di P1. Rasio ini paling jauh bisa menjadi *fitur* (input) untuk model nanti. Latihan ini juga tidak menghasilkan keputusan kredit.
- **Bantuan AI:** Alat: Claude (Anthropic). Tujuan: membantu menjalankan notebook, menyusun cell rekap, dan membuat draf jawaban. Bagian yang dibantu: cell "Rekap bukti" dan draf seluruh jawaban di atas. Cara memeriksa: saya menjalankan ulang semua cell di Google Colab (Runtime > Run all), mencocokkan `df.shape` (8, 3) dan duplikat A002 dengan tabel, menghitung manual rasio A001 (2.000.000 ÷ 8.000.000 = 25%; 2.800.000 ÷ 8.000.000 = 35%), memilih A003, A005, dan A006 satu per satu di formulir untuk melihat pesannya, lalu menulis ulang penjelasan dengan kata-kata sendiri.

### Simpan dan kumpulkan tautan
Simpan code, output, dan jawaban di salinan Drive. Bagikan kepada dosen sebagai **Viewer**, lalu kirim tautan salinan Anda ke LMS. Tidak perlu ZIP, PDF jawaban, screenshot, atau CSV terpisah. Kuis 2 formatif dikerjakan terpisah. Tenggat mengikuti LMS.


## Rujukan dan batas penggunaan
- RPS resmi prodi, Artificial Intelligence in Finance, 21 Oktober 2026. Sesi 1: peran AI/ML dan kuis. Sesi 2: lingkungan Python/data dan praktikum. Eksplorasi/praproses mendalam dilanjutkan sesi 3.
- Google Colab: https://research.google.com/colaboratory/faq.html
- Python: https://docs.python.org/3/tutorial/introduction.html
- pandas: https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
- Google Drive sharing: https://support.google.com/drive/answer/2494822

Notebook dan contoh disusun dengan bantuan AI, kemudian kode serta perhitungannya diperiksa. Mahasiswa tetap menjelaskan pemahamannya sendiri dan mendeklarasikan bantuan AI. Data latihan tidak memuat identitas nasabah atau data perusahaan asli.
